# Улучшение семантических кодов

Продолжение [semantic_codes.ipynb](semantic_codes.ipynb). Там коды сами по себе дали nDCG@10 ≈ 0.06–0.13. Видно две причины:

- **Код отражает контекст, а не слово.** Соседние слова одного предложения получают почти одинаковые коды: старшие главные компоненты векторов ColBERT кодируют тему предложения.
- **Запросы и документы лежат в разных областях.** Короткий вопрос и абзац энциклопедии модель кодирует со сдвигом, поэтому старшие биты у них систематически разные.

Что проверяем (всё на ITQ, 12 бит, 4 хеша — лучшая зона из первого ноутбука):

1. Убрать стоп-слова.
2. Вычесть средний вектор предложения или текста из вектора слова.
3. Сдвинуть облако слов запросов к облаку слов документов.
4. Пропустить первые главные компоненты (skip_top).
5. Комбинации, затем сетка по числу бит и хешей для лучшего варианта, затем гибрид.

Метрика `overlap_gap` — насколько чаще коды запроса встречаются в релевантном документе, чем в случайном. Если она около нуля, коды не различают документы.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

import semantic_codes as sc

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 250)

MODEL = "BAAI/bge-m3"
DATASET = "ai-forever/rubq-retrieval"
N_QUERIES, N_DISTRACTORS = 300, 2000
CACHE_DIR = "cache"

BASE = dict(n_bits=12, n_hashes=4, method="itq")   # общая конфигурация для сравнения вариантов
COLS = ["variant", "n_bits", "n_hashes", "skip_top", "nDCG@10", "Recall@10", "Recall@100", "MRR", "overlap_gap", "vocab"]

In [2]:
data = sc.load_mteb_retrieval(DATASET, N_QUERIES, N_DISTRACTORS)
print(data)

tag = f"{CACHE_DIR}/{data.name.replace('/', '_')}_{N_QUERIES}_{N_DISTRACTORS}_{MODEL.replace('/', '_')}"
# векторы токенов ColBERT здесь не нужны (keep_tokens=False) — отдельный, лёгкий кеш
encoder = sc.TokenEncoder(MODEL)
doc_enc = sc.encode_cached(lambda: encoder, data.docs, f"{tag}_docs_words.pkl", keep_tokens=False)
q_enc = sc.encode_cached(lambda: encoder, data.queries, f"{tag}_queries_words.pkl", keep_tokens=False)
del encoder
print("слов в корпусе:", len(doc_enc.word_vecs), "| в запросах:", len(q_enc.word_vecs))

RetrievalData(ai-forever/rubq-retrieval: 2487 docs, 300 queries, 487 qrels)


[тексты] 32/2487  2s

[тексты] 672/2487  4s

[тексты] 1312/2487  10s

[тексты] 1952/2487  21s


[предложения] 32/8692  62s

... (12 outputs left out)

[предложения] 8352/8692  87s

[тексты] 32/300  0s


[предложения] 32/300  0s

слов в корпусе: 155479 | в запросах: 1846


## Базовые точки: BM25 по словам (после исправления ударений) и коды как есть

In [3]:
lem_d = [sc.lemmatize(t) for t in data.docs]
lem_q = [sc.lemmatize(t) for t in data.queries]
S_words = sc.BM25().fit(lem_d).score_matrix(lem_q)
S_dense = sc.dense_scores(q_enc.dense, doc_enc.dense)
base = pd.DataFrame({"BM25 по словам": sc.evaluate(S_words, data.qrels), "BGE-M3 dense": sc.evaluate(S_dense, data.qrels)}).T
base

,Recall@1,Recall@5,Recall@10,Recall@100,nDCG@10,MRR
BM25 по словам,0.4874,0.8215,0.8700,0.9667,0.7640,0.7639
BGE-M3 dense,0.6896,0.9479,0.9659,0.9922,0.9323,0.9424


## Варианты подготовки векторов слов

Каждый вариант — функция, возвращающая (векторы документов, векторы запросов, параметры бинаризатора).
Варианты строятся по одному, чтобы не держать в памяти десяток копий корпуса.
В таблицу идут только метрики, тяжёлые поля (токены кодов) сохраняются лишь для лучшего результата.

In [4]:
d_stop, q_stop = sc.filter_words(doc_enc), sc.filter_words(q_enc)
print(f"после удаления стоп-слов осталось {len(d_stop.word_vecs) / len(doc_enc.word_vecs):.0%} слов корпуса")

variants = {
    "как есть": lambda: (doc_enc, q_enc, {}),
    "без стоп-слов": lambda: (d_stop, q_stop, {}),
    "центр. по предложению": lambda: (sc.center_by_group(doc_enc), sc.center_by_group(q_enc), {}),
    "центр. по предложению ×0.5": lambda: (sc.center_by_group(doc_enc, alpha=0.5), sc.center_by_group(q_enc, alpha=0.5), {}),
    "центр. по тексту": lambda: (sc.center_by_group(doc_enc, "doc"), sc.center_by_group(q_enc, "doc"), {}),
    "сдвиг запросов к документам": lambda: (doc_enc, sc.align_queries(doc_enc, q_enc), {}),
    "skip_top=2": lambda: (doc_enc, q_enc, {"skip_top": 2}),
    "skip_top=8": lambda: (doc_enc, q_enc, {"skip_top": 8}),
    "без стоп-слов + центр. по предл.": lambda: (sc.center_by_group(d_stop), sc.center_by_group(q_stop), {}),
    "без стоп-слов + центр. по предл. + сдвиг": lambda: (lambda d, q: (d, sc.align_queries(d, q), {}))(sc.center_by_group(d_stop), sc.center_by_group(q_stop)),
}

после удаления стоп-слов осталось 77% слов корпуса


In [5]:
rows, best, raw_codes = [], None, None
for name, make in variants.items():
    d, q, kw = make()
    r = sc.codes_experiment(d, q, data.qrels, label=name, **BASE, **kw)
    print(f"{name:45s} nDCG@10={r['nDCG@10']:.4f}  gap={r['overlap_gap']:.4f}")
    if name == "как есть":
        raw_codes = r
    if best is None or r["nDCG@10"] > best["nDCG@10"]:
        best = r
    rows.append(sc.strip_heavy([r])[0])
    del d, q, r
res = pd.DataFrame(rows).reindex(columns=COLS)
res.sort_values("nDCG@10", ascending=False)

как есть                                      nDCG@10=0.1258  gap=0.1611


без стоп-слов                                 nDCG@10=0.1285  gap=0.1486


центр. по предложению                         nDCG@10=0.0369  gap=0.0576


центр. по предложению ×0.5                    nDCG@10=0.0962  gap=0.1306


центр. по тексту                              nDCG@10=0.0172  gap=0.0333


сдвиг запросов к документам                   nDCG@10=0.1081  gap=0.1511


skip_top=2                                    nDCG@10=0.1183  gap=0.1176


skip_top=8                                    nDCG@10=0.0992  gap=0.1069


без стоп-слов + центр. по предл.              nDCG@10=0.0460  gap=0.0632


без стоп-слов + центр. по предл. + сдвиг      nDCG@10=0.0430  gap=0.0631


,variant,n_bits,n_hashes,skip_top,nDCG@10,Recall@10,Recall@100,MRR,overlap_gap,vocab
1,без стоп-слов,12,4,NaN,0.1285,0.2089,0.5463,0.1306,0.1486,16361
0,как есть,12,4,NaN,0.1258,0.1936,0.5454,0.1307,0.1611,16378
6,skip_top=2,12,4,2.0000,0.1183,0.2069,0.5219,0.1258,0.1176,16383
5,сдвиг запросов к документам,12,4,NaN,0.1081,0.1792,0.5535,0.1233,0.1511,16378
7,skip_top=8,12,4,8.0000,0.0992,0.1849,0.4979,0.0975,0.1069,16384
3,центр. по предложению ×0.5,12,4,NaN,0.0962,0.1672,0.4823,0.0960,0.1306,16238
8,без стоп-слов + центр. по предл.,12,4,NaN,0.0460,0.0660,0.2257,0.0614,0.0632,15786
9,без стоп-слов + центр. по предл. + сдвиг,12,4,NaN,0.0430,0.0599,0.2396,0.0612,0.0631,15786
2,центр. по предложению,12,4,NaN,0.0369,0.0646,0.2127,0.0426,0.0576,15857
4,центр. по тексту,12,4,NaN,0.0172,0.0294,0.1269,0.0241,0.0333,16354


### Как выглядят коды лучшего варианта

In [6]:
d_best, q_best, kw_best = variants[best["variant"]]()
print("лучший вариант:", best["variant"])
n_h = best["n_hashes"]
for name, r, q_words, d_words in [("как есть", raw_codes, q_enc.words, doc_enc.words), (best["variant"], best, q_best.words, d_best.words)]:
    rel = next(iter(data.qrels[0]))
    print(f"--- {name}")
    print(sc.coded_text(q_words[0], r["query_codes"][0], [], n_h))
    print(sc.coded_text(d_words[rel][:15], r["doc_codes"][rel], [], n_h))

лучший вариант: без стоп-слов
--- как есть
Как§h0_001111101010§§h1_001000001010§§h2_110001000111§§h3_011110001110§ зовут§h0_001111100010§§h1_001000001011§§h2_110011000110§§h3_011100001110§ Бречалова§h0_111111101010§§h1_001000001110§§h2_110000100110§§h3_011010001010§ Удмуртия§h0_001111101110§§h1_111000101010§§h2_110000000101§§h3_111010101100§
Александр§h0_111110101010§§h1_001000011000§§h2_110000100010§§h3_111010111010§ Владимирович§h0_111100101010§§h1_001000011000§§h2_110000100010§§h3_111010101010§ Бречалов§h0_111110101010§§h1_001010001000§§h2_110000100010§§h3_111010101010§ род§h0_111110110010§§h1_011000011000§§h2_110001100010§§h3_111000101010§ 18§h0_011110011010§§h1_011000011011§§h2_110111000010§§h3_111000001110§ ноября§h0_011110010010§§h1_011000011011§§h2_110101000010§§h3_111100001110§ 1973§h0_011110010010§§h1_011000011011§§h2_110110000010§§h3_111100001110§ п§h0_011111011010§§h1_011000001011§§h2_111100000010§§h3_111100101011§ Тлюстенхабль§h0_011110010010§§h1_011000001011§§h2_111100100

## Сетка бит и хешей для лучшего варианта

In [7]:
grid_rows, best_codes = [], best
for nb in [10, 12, 14, 16, 20]:
    for nh in [1, 4, 8]:
        r = sc.codes_experiment(d_best, q_best, data.qrels, n_bits=nb, n_hashes=nh, method="itq", label=best["variant"], **kw_best)
        print(f"{nb:2d} бит, {nh} хеш: nDCG@10={r['nDCG@10']:.4f}  Recall@100={r['Recall@100']:.4f}")
        if r["nDCG@10"] > best_codes["nDCG@10"]:
            best_codes = r
        grid_rows.append(sc.strip_heavy([r])[0])
        del r
grid = pd.DataFrame(grid_rows)
grid.pivot_table(index="n_bits", columns="n_hashes", values="nDCG@10")

10 бит, 1 хеш: nDCG@10=0.0488  Recall@100=0.3848


10 бит, 4 хеш: nDCG@10=0.1010  Recall@100=0.5321


10 бит, 8 хеш: nDCG@10=0.1147  Recall@100=0.5558


12 бит, 1 хеш: nDCG@10=0.0894  Recall@100=0.4202


12 бит, 4 хеш: nDCG@10=0.1285  Recall@100=0.5463


12 бит, 8 хеш: nDCG@10=0.1513  Recall@100=0.6089


... (6 outputs left out)

20 бит, 1 хеш: nDCG@10=0.0617  Recall@100=0.1222


20 бит, 4 хеш: nDCG@10=0.1515  Recall@100=0.2839


20 бит, 8 хеш: nDCG@10=0.1940  Recall@100=0.3783


n_hashes,1,4,8
n_bits,,,
10,0.0488,0.1010,0.1147
12,0.0894,0.1285,0.1513
14,0.0787,0.1364,0.1707
16,0.0721,0.1301,0.1824
20,0.0617,0.1515,0.1940


In [8]:
grid.pivot_table(index="n_bits", columns="n_hashes", values="Recall@100")

n_hashes,1,4,8
n_bits,,,
10,0.3848,0.5321,0.5558
12,0.4202,0.5463,0.6089
14,0.2811,0.5251,0.6062
16,0.2103,0.4655,0.5888
20,0.1222,0.2839,0.3783


## Гибрид: коды + сущности + слова

Сравниваем коды из первого ноутбука («как есть») и лучшие коды отсюда в одном и том же гибриде.

In [9]:
fields_d = {"entities": [sc.extract_entities(t) for t in data.docs], "lemmas": lem_d}
fields_q = {"entities": [sc.extract_entities(t) for t in data.queries], "lemmas": lem_q}

hyb_rows = []
for name, r in [("коды как есть", raw_codes), (f"лучшие коды ({best_codes['n_bits']} бит, {best_codes['n_hashes']} хеш)", best_codes)]:
    fd, fq = {**fields_d, "codes": r["doc_codes"]}, {**fields_q, "codes": r["query_codes"]}
    FS = sc.MultiFieldBM25({"codes": 1, "entities": 1, "lemmas": 1}).fit(fd).field_scores(fq)
    for w_c in [0, 0.25, 0.5, 1.0]:
        for w_e in [0, 1.0]:
            w = {"codes": w_c, "entities": w_e, "lemmas": 2.0}
            hyb_rows.append({"коды": name, **w, **sc.evaluate(sc.combine_scores(FS, w), data.qrels)})
hyb = pd.DataFrame(hyb_rows)
hyb.sort_values("nDCG@10", ascending=False).head(12)

,коды,codes,entities,lemmas,Recall@1,Recall@5,Recall@10,Recall@100,nDCG@10,MRR
13,"лучшие коды (20 бит, 8 хеш)",0.5000,1.0000,2.0000,0.5280,0.8379,0.8922,0.9728,0.7936,0.7968
11,"лучшие коды (20 бит, 8 хеш)",0.2500,1.0000,2.0000,0.5219,0.8388,0.8894,0.9728,0.7913,0.7945
3,коды как есть,0.2500,1.0000,2.0000,0.5246,0.8290,0.8903,0.9744,0.7890,0.7986
10,"лучшие коды (20 бит, 8 хеш)",0.2500,0.0000,2.0000,0.5216,0.8343,0.8742,0.9717,0.7828,0.7908
12,"лучшие коды (20 бит, 8 хеш)",0.5000,0.0000,2.0000,0.5210,0.8327,0.8783,0.9717,0.7815,0.7896
2,коды как есть,0.2500,0.0000,2.0000,0.5194,0.8163,0.8869,0.9725,0.7774,0.7892
1,коды как есть,0.0000,1.0000,2.0000,0.4910,0.8293,0.8869,0.9694,0.7741,0.7696
9,"лучшие коды (20 бит, 8 хеш)",0.0000,1.0000,2.0000,0.4910,0.8293,0.8869,0.9694,0.7741,0.7696
0,коды как есть,0.0000,0.0000,2.0000,0.4874,0.8215,0.8700,0.9667,0.7640,0.7639
8,"лучшие коды (20 бит, 8 хеш)",0.0000,0.0000,2.0000,0.4874,0.8215,0.8700,0.9667,0.7640,0.7639


In [10]:
summary = pd.concat([
    base,
    pd.DataFrame({"коды как есть (12 бит, 4 хеш)": {k: raw_codes[k] for k in base.columns},
                  f"лучшие коды: {best_codes['variant']} ({best_codes['n_bits']} бит, {best_codes['n_hashes']} хеш)": {k: best_codes[k] for k in base.columns}}).T,
    hyb.sort_values("nDCG@10", ascending=False).groupby("коды").head(1).set_index("коды")[base.columns].rename(index=lambda s: "гибрид: " + s),
])
summary.sort_values("nDCG@10", ascending=False)

,Recall@1,Recall@5,Recall@10,Recall@100,nDCG@10,MRR
BGE-M3 dense,0.6896,0.9479,0.9659,0.9922,0.9323,0.9424
"гибрид: лучшие коды (20 бит, 8 хеш)",0.5280,0.8379,0.8922,0.9728,0.7936,0.7968
гибрид: коды как есть,0.5246,0.8290,0.8903,0.9744,0.7890,0.7986
BM25 по словам,0.4874,0.8215,0.8700,0.9667,0.7640,0.7639
"лучшие коды: без стоп-слов (20 бит, 8 хеш)",0.1036,0.2139,0.2753,0.3783,0.1940,0.2045
"коды как есть (12 бит, 4 хеш)",0.0553,0.1506,0.1936,0.5454,0.1258,0.1307
